# Laboratorio L8 - Analisi automatica degli URL

B.4 - Cybersicurezza e IA

Il notebook contiene una funzione che scompone un URL e segnala le caratteristiche sospette, come fa, in forma molto più completa, un filtro antiphishing. Tutti gli URL sono esempi con domini `.example`, che non esistono.

Eseguire le celle in ordine con `Maiusc+Invio`.

## 1. Scomporre un URL

- `urlsplit(url)`: scompone l'URL; le parti si leggono con `.scheme`, `.hostname`, `.path`, `.query`
- `.hostname` restituisce il nome host in minuscolo, senza l'eventuale parte che precede `@`
- `.username`: la parte prima di `@`, se presente

In [ ]:
from urllib.parse import urlsplit, unquote

url = "https://www.bancaaurora.example@sicuro-login.example/aurora?x=1#top"
parti = urlsplit(url)
print("schema    :", parti.scheme)
print("nome host :", parti.hostname)
print("utente    :", parti.username)
print("percorso  :", parti.path)
print("parametri :", parti.query)

## 2. Dominio registrato

Regola semplificata: gli ultimi due elementi del nome host. Per alcuni suffissi (per esempio `.gov.it`, `.co.uk`) il dominio registrato ha tre elementi: il programma usa un piccolo elenco di questi suffissi. I programmi professionali usano la Public Suffix List completa.

- `host.split(".")`: lista delle parti separate dal punto
- `parti[-3:]`, `parti[-2:]`: ultimi tre o due elementi
- `".".join(...)`: ricompone la stringa

In [ ]:
SUFFISSI_DOPPI = {"gov.it", "edu.it", "co.uk", "ac.uk", "com.au"}

def dominio_registrato(host):
    parti = host.split(".")
    if ".".join(parti[-2:]) in SUFFISSI_DOPPI:
        return ".".join(parti[-3:])
    return ".".join(parti[-2:])

for h in ["www.bancaaurora.example", "bancaaurora.example.verifica-accessi.example",
          "www.miur.gov.it", "www.bbc.co.uk"]:
    print(f"{h:48} -> {dominio_registrato(h)}")

## 3. La funzione di analisi

`analizza_url(url, dominio_ufficiale)` confronta l'URL con il dominio ufficiale dell'organizzazione attesa e restituisce l'elenco dei segnali trovati.

Segnali controllati:

- schema `http` (non cifrato)
- presenza di `@` nel nome host
- indirizzo IP al posto del nome
- Punycode (`xn--`) o caratteri non ASCII nel nome host (possibile omografo)
- dominio registrato diverso da quello ufficiale
- nome dell'organizzazione presente nell'URL ma fuori dal dominio registrato (sottodominio fuorviante, nome nel percorso)
- dominio che contiene il nome ufficiale con aggiunte (per esempio `accesso-bancaaurora`)
- servizi noti di reindirizzamento o accorciamento, con decodifica della destinazione se presente nei parametri

Funzioni usate:

- `re.fullmatch(r"[0-9.]+", host)`: vero se il nome host è fatto solo di cifre e punti (indirizzo IPv4)
- `host.isascii()`: vero se tutti i caratteri sono ASCII
- `unquote(testo)`: decodifica i caratteri `%xx` (come `URL Decode` di CyberChef)
- `testo.encode("idna").decode()` e `testo.encode().decode("idna")`: conversione verso e da Punycode

In [ ]:
import re
from urllib.parse import parse_qs

REINDIRIZZATORI = {"redirect.example", "short-link.example"}

def analizza_url(url, dominio_ufficiale):
    segnali = []
    p = urlsplit(url)
    host = p.hostname or ""
    nome = dominio_ufficiale.split(".")[0]              # es. "bancaaurora"

    if p.scheme == "http":
        segnali.append("connessione non cifrata (http)")
    if p.username:
        segnali.append(f"'@' nell'indirizzo: la parte '{p.username}' viene ignorata")
    if re.fullmatch(r"[0-9.]+", host):
        segnali.append("indirizzo IP al posto del nome")
    if not host.isascii():
        segnali.append(f"caratteri non latini nel nome: in Punycode {host.encode('idna').decode()}")
        host = host.encode("idna").decode()
    if "xn--" in host:
        segnali.append(f"nome in Punycode, forma leggibile: {host.encode().decode('idna')}")

    reg = dominio_registrato(host)
    if re.fullmatch(r"[0-9.]+", host):
        if nome in url.lower():
            segnali.append(f"il nome '{nome}' compare nell'URL ma il sito è un indirizzo IP")
    elif reg in REINDIRIZZATORI:
        segnali.append("servizio di reindirizzamento o accorciamento: la destinazione è nascosta")
        for valori in parse_qs(p.query).values():
            for v in valori:
                if v.startswith("http"):
                    segnali.append(f"destinazione dichiarata nei parametri: {unquote(v)}")
    elif reg != dominio_ufficiale:
        segnali.append(f"dominio registrato '{reg}' diverso da '{dominio_ufficiale}'")
        if nome in url.lower():
            segnali.append(f"il nome '{nome}' compare nell'URL ma fuori dal dominio registrato")

    giudizio = "SOSPETTO" if segnali else "nessun segnale trovato"
    return giudizio, segnali

def mostra(url, dominio_ufficiale):
    giudizio, segnali = analizza_url(url, dominio_ufficiale)
    print(url)
    print("  ->", giudizio)
    for s in segnali:
        print("     -", s)
    print()

## 4. Gli URL del corpus (esercizio 2)

In [ ]:
CORPUS = [
    ("https://spediveloce.consegna-pacchi.example/r?id=IT4471", "spediveloce.example"),
    ("https://redirect.example/go?to=https%3A%2F%2Fwebmail.istituto-esempio.example.rinnovo-casella.example%2Flogin", "istituto-esempio.example"),
    ("https://bаncaaurora.example/annulla", "bancaaurora.example"),
    ("https://paga-parcheggio-roma.example/sosta?zona=12", "comune-esempio.example"),
    ("https://www.bancaaurora.example/login", "bancaaurora.example"),
    ("https://app.bancaaurora.example/", "bancaaurora.example"),
]
for url, ufficiale in CORPUS:
    mostra(url, ufficiale)

Nel terzo URL la prima `a` di `bаncaaurora` è una lettera cirillica: a occhio non si distingue.

## 5. Nuovi URL (esercizio 3)

Aggiungere alla lista tre URL inventati, sempre con domini `.example`: uno legittimo e due ingannevoli con tecniche diverse. Poi cercare un URL ingannevole che il programma non riconosce.

In [ ]:
MIEI_URL = [
    ("https://www.bancaaurora.example/aiuto", "bancaaurora.example"),
    ("http://192.0.2.44/bancaaurora/login", "bancaaurora.example"),
    # aggiungere qui altri URL, nella forma ("url", "dominio ufficiale"),
]
for url, ufficiale in MIEI_URL:
    mostra(url, ufficiale)

**Domande**

- Quale URL ingannevole avete trovato che il programma non riconosce? Perché?
- Il programma dice "nessun segnale trovato" per un URL del dominio ufficiale. Questo garantisce che la pagina sia sicura? Pensare al caso di un sito legittimo compromesso o di un messaggio legittimo inviato da una casella compromessa.

Risposte: